# Ball position from frozen I-JEPA tokens: increasing-radius experiment

Select **GPU → Run all**. This is separate from ranking and distance-MSE experiments.

**Frozen spatial tokens → position readout → predicted (x,y).**

The model applies a shared32channel projection to each patch token, retains patch order, then uses a64unit hidden layer to predict two coordinates. Outputs are unrestricted. It receives no physical coordinates or goal tokens as inputs.

Coordinate labels are normalized as `(true_xy - [50,42])/34`; the center and scale stay fixed at every stage. Loss is mean coordinate squared error. Results are converted back to simulation units.

Same hand-positioned training rays/radii and cumulative limits **6 → 14 → 22 → 34**, giving48,90,105,115training positions. The first model is fresh; later fits keep earlier points and warm start previous weights. Each stage uses500Adam updates,32positions/update,lr1e-4 with fresh optimizer moments.

Primary result: **Euclidean position error**, not just distance-to-goal error. Good distance alone can hide a ball placed in the wrong direction. Coordinate supervision is richer than the earlier distance-only labels.

All probes are fixed before fitting. New timestamped outputs go under `radial_position_v1`; existing experiments are preserved. No planning/actions/world-model checkpoint.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_position_v1' / RUN_ID
CACHE = Path('/content/radial-position-cache') / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-position')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = None
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-position-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Local feature cache:', CACHE)


## Frozen encoder and renderer

Only the existing I-JEPA encoder checkpoint in Drive is needed. Each stage fits the position head on cached frozen tokens. Old adapters and experiment outputs are not used. The source pin makes this run independent of later branch changes.


In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
assert shutil.disk_usage('/content').free > 7*2**30, 'At least 7 GiB free local disk is needed for the float32 feature cache.'
print('Ready. No world-model checkpoint or previous experiment folder is required.')

def stage_command(operation,index=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_experiment',operation,
          '--output-dir',OUTPUT,'--cache-dir',CACHE,'--device','cuda']
    if operation=='prepare': args+=['--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--objective','position']
    if index is not None: args+=['--stage-index',str(index)]
    return args


## Encode the fixed design once

Freshly render and encode all4,668positions, including future training points and probes. Pre-encoding does not expose them to fitting: the optimizer receives only that stage's approved coordinate labels and features.

About5.7GiB of float32 token shards stay on local Colab disk. Persistent configurations, training features, predictions, checkpoints, reports and figures go to Drive. The reference goal-frame tokens support raw-MSE diagnostics only; **the position head never consumes that reference image or tokens**.


In [ ]:
started=time.monotonic()
shell(stage_command('prepare'),cwd=REPO)
design=json.loads((OUTPUT/'design.json').read_text())
config=json.loads((OUTPUT/'config.json').read_text())
print(f'Fresh encoding took {(time.monotonic()-started)/60:.1f} minutes.')
print('Unique encoded images:',len(design['points']))
print('Stage limits and cumulative training positions:',[(s['outer_radius'],len(s['training_ids'])) for s in design['stages']])
print('Matched offset families:',len(design['matched_family_ids']))
assert config['kind']=='radial_position_v1'
print('Fitting:',config['fitting'],'| coordinate regression only')
print('Fixed label center/scale:',config['position_center'],config['position_scale'])
print('Reference layout/goal frame (not an input to the position fitter):')
from IPython.display import display,Image
display(Image(filename=str(OUTPUT/'goal.png')))


## Fit, evaluate and plot each stage

- **Position error:** mean Euclidean error between predicted and true (x,y). Position RMSE is `sqrt(mean(||pred_xy-true_xy||**2))`, not per-coordinate RMSE. Units are physical simulation units.
- **Constant baseline:** always predict the mean of the current training coordinates. It is fitted only on offered training labels.
- **Distance MAE:** error in Euclidean distance from predicted (x,y) to the known fixed goal. This is secondary to position error.
- Heldout groups exclude exact coordinate and rendered-image overlap with offered training points. Directional positions are deduplicated.
- `inside`/`outside` position groups use true radius. Additional outside shells show distance beyond the current training boundary.
- The fixed unseen-ray R<=6 group stays the same across stages, revealing damage to earlier generalization.
- Derived-distance uphill/interpolation/offset tests measure whether moving toward the goal reduces the distance calculated from predicted coordinates. This cost is **not adapted latent MSE**. Raw MSE is a directional reference only.
- Maps span the full right compartment to x≈60. True/predicted distance maps share a scale; position-error maps use their own full range. Predictions are never clipped to the training region or physical box.
- Source/config/cache/checkpoint lineage is validated before fitting or reuse. A completed stage can regenerate missing figures without retraining.

At the final radius34 stage, directional extrapolation groups are empty and reported N/A. Grid corners beyond34 still appear in the maps. These are descriptive, spatially correlated probes.


In [ ]:
def number(value): return 'N/A' if value is None else f'{value:.4f}'
def pct(value): return 'N/A' if value is None else f'{value:.1%}'

def show_report(r):
    t=r['training'];p=r['position']
    print(f"\nPOSITION STAGE {r['stage']} | R={r['outer_radius']} | {r['training_positions']} training positions")
    print('Initialization:',r['initialization'],'| model shape:',t['model_shape'])
    print('Actual sampled observations:',sum(t['sample_counts']),'| primitive actions:',r['primitive_interactions'])
    print('Full-training normalized coordinate loss:',t['full_position_loss_before'],'→',t['full_position_loss_after'])
    print('Training mean position error:',t['position_error_before'],'→',t['position_error_after'])
    print('Baseline always predicts current training-label mean:',p['constant_position'])
    print('\nPOSITION METRICS: physical simulation units')
    print(f"{'cohort':>24} {'count':>7} {'method':>9} {'mean error':>12} {'pos RMSE':>10} {'x MAE':>9} {'y MAE':>9} {'dist MAE':>10}")
    groups=[('training',p['training']),('heldout inside',p['directional']['inside']),('heldout outside',p['directional']['outside']),
            ('unseen rays inside',p['withheld_direction']['inside']),('unseen rays outside',p['withheld_direction']['outside']),
            ('fixed unseen R<=6',p['fixed_near_withheld_direction'])]
    for label,g in groups:
        for method in ('constant','model'):
            m=g[method];xy=m['xy_mae'] if m['xy_mae'] is not None else [None,None]
            print(f"{label:>24} {g['count']:>7} {method:>9} {number(m['mean_position_error']):>12} "
                  f"{number(m['position_rmse']):>10} {number(xy[0]):>9} {number(xy[1]):>9} {number(m['distance_mae']):>10}")
    print('\nOUTSIDE SHELLS: genuinely heldout directional positions')
    for shell in p['outside_shells']:
        upper='infinity' if shell['upper'] is None else shell['upper']
        g=shell['metrics']
        print(f"Beyond boundary ({shell['lower']},{upper}] | n={g['count']} | "
              f"position error constant={number(g['constant']['mean_position_error'])}, model={number(g['model']['mean_position_error'])}")
    checks=r['spatial_checks']
    print('\nDERIVED-DISTANCE DIRECTIONAL TEST: uphill is wrong while moving TOWARD the goal')
    print(f"{'directions':>20} {'region':>9} {'method':>18} {'steps':>7} {'uphill':>9} {'heldout up/steps':>20} {'ties':>7}")
    for kind,regions in checks['generalization'].items():
        for region,methods in regions.items():
            for method,m in methods.items():
                label='raw MSE' if method=='raw' else 'position distance'
                print(f"{kind:>20} {region:>9} {label:>18} {m['comparisons']:>7} {pct(m['up_fraction']):>9} "
                      f"{m['heldout_up_steps']:>8}/{m['heldout_comparisons']:<11} {m['tie_steps']:>7}")
    for key,title in (('offset_fitted_centers','OFFERED TRAINING ENDPOINTS'),('offset_summary','FIXED ALL-RADII COHORT')):
        print('\nMATCHED OFFSETS:',title)
        print(f"{'offset':>7} {'families':>9} {'method':>18} {'endpoint OK':>12} {'monotone':>10} {'uphill':>9} {'flat':>8}")
        for row in checks[key]:
            for method in ('raw','adapted'):
                label='raw MSE' if method=='raw' else 'position distance';m=row[method]
                print(f"{row['offset']:>7g} {row['families']:>9} {label:>18} {pct(m['endpoint_correct']):>12} "
                      f"{pct(m['informative_monotone']):>10} {pct(m['up_fraction']):>9} {pct(m['flat']):>8}")
    print('No coordinate/image overlap with offered fitting inputs:',sum(r['exposure']['heldout_from_offered_training']),'/',len(design['points']))
    print('Goal reference is not a fitting input. Predicted coordinates and all errors are saved in report.json.')

reports=[]
for index in (1,2,3,4):
    started=time.monotonic()
    shell(stage_command('stage',index),cwd=REPO)
    folder=OUTPUT/f'stages/s{index}'
    report=json.loads((folder/'report.json').read_text());reports.append(report)
    show_report(report)
    print(f'Stage took {(time.monotonic()-started)/60:.1f} minutes.')
    for name in ('position_maps','coordinate_predictions','radial_distances','local_generalization','training_loss'):
        display(Image(filename=str(folder/'figures'/f'{name}.png')))


## Compare stages on position extrapolation

Lower mean position error is better. Compare the model and constant baseline on the same rows. Inside/outside populations change as the training radius expands; the fixed-near table holds evaluation positions constant. Radius34 outside is N/A, not zero error.


In [ ]:
print('SUMMARY: mean Euclidean position error on genuinely unseen directions')
print(f"{'stage':>6} {'R':>5} {'inside const':>14} {'inside model':>14} {'outside const':>14} {'outside model':>14}")
for r in reports:
    g=r['position']['withheld_direction']
    print(f"{r['stage']:>6} {r['outer_radius']:>5} "
          f"{number(g['inside']['constant']['mean_position_error']):>14} {number(g['inside']['model']['mean_position_error']):>14} "
          f"{number(g['outside']['constant']['mean_position_error']):>14} {number(g['outside']['model']['mean_position_error']):>14}")
print('\nFIXED NEAR GROUP: same unseen directions at R<=6 after every stage')
print(f"{'stage':>6} {'positions':>10} {'position error':>16} {'position RMSE':>16} {'distance MAE':>14}")
for r in reports:
    g=r['position']['fixed_near_withheld_direction'];m=g['model']
    print(f"{r['stage']:>6} {g['count']:>10} {number(m['mean_position_error']):>16} "
          f"{number(m['position_rmse']):>16} {number(m['distance_mae']):>14}")
print('\nConstant baseline changes with the current training-label mean. Fixed-near model rows use the same positions.')
print('Inside/outside groups otherwise change with training radius; counts are in detailed reports.')
print('No outside-directional probes remain at R34; grid corners beyond34 still appear in maps.')
print('Results:',OUTPUT)


## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. All four position checkpoints, reports with predicted coordinates, and five PNG/PDF figures per stage are retained in the new Drive folder.

A failed extrapolation test would concern this readout and training coverage; it would not establish that I-JEPA lacks position information. This uses coordinate supervision, which is richer than distance-only labels. Actual GPU/Colab behavior and scientific outcomes are established by your run; local verification uses synthetic features.
